# 03 — CU1: clasificación por imagen (ResNet-50 + EfficientNet-B0) y fusión

Ajusta dos arquitecturas por transfer learning (primero solo la cabeza,
después descongelando las últimas capas), las compara, y evalúa una
fusión imagen+texto (reutilizando el BETO de 02_cu2_beto) con el peso
elegido en validación. Meta (OE1): accuracy ≥ 0,80 y F1-macro ≥ 0,75 en
el conjunto de prueba.

Tiempo esperado en GPU T4: 30-50 minutos para el dataset completo (dos
arquitecturas, dos fases cada una). Es, de los 4 notebooks, el más lento
— si Colab se desconecta, los checkpoints intermedios quedan en Drive
para retomar (ver la celda de guardado de cada fase).

In [ ]:
# Si ya corriste esta celda en esta sesión de Colab, puedes saltarla.
import sys

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA_ML = "/content/drive/MyDrive/NovaStore_IA/ml"
    get_ipython().system(f"pip install -q -r {{CARPETA_ML}}/requirements.txt")
else:
    # Ejecución local (verificación, no Colab): usa la carpeta ml/ del repo.
    CARPETA_ML = str(__import__("pathlib").Path.cwd())

sys.path.insert(0, CARPETA_ML)
print("CARPETA_ML =", CARPETA_ML)


In [ ]:
import random
from pathlib import Path

import numpy as np
import torch

# --- Configuración: edita esto antes de correr en Colab ---
CARPETA_DRIVE = Path(CARPETA_ML) / "datos"  # train.csv/val.csv/test.csv (los crea 00_preparar_dataset.ipynb)
CARPETA_MODELOS = Path(CARPETA_ML) / "modelos" / "cu1-imagen"
MODO_RAPIDO = False  # True = 5 ejemplos/clase, 1 época — solo para probar que el notebook corre de punta a punta
SEMILLA = 42
# ------------------------------------------------------------

random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
print(f"MODO_RAPIDO={MODO_RAPIDO} · dispositivo={DISPOSITIVO} · GPU disponible: {torch.cuda.is_available()}")
if DISPOSITIVO == "cpu":
    print("Sin GPU: en Colab, activa Entorno de ejecución -> Cambiar tipo de entorno -> T4 GPU.")

CARPETA_MODELOS.mkdir(parents=True, exist_ok=True)


## 1. Cargar la partición y armar los `Dataset` de imágenes

In [ ]:
import pandas as pd

if MODO_RAPIDO:
    from _modo_rapido_fallback import generar_si_falta
    generar_si_falta(CARPETA_DRIVE)


train = pd.read_csv(CARPETA_DRIVE / "train.csv")
val = pd.read_csv(CARPETA_DRIVE / "val.csv")
test = pd.read_csv(CARPETA_DRIVE / "test.csv")

if MODO_RAPIDO:
    train = train.groupby("categoria", group_keys=False).head(5)
    val = val.groupby("categoria", group_keys=False).head(2)
    test = test.groupby("categoria", group_keys=False).head(2)

print(f"train={len(train)} val={len(val)} test={len(test)}")
train.head()


In [ ]:
from io import BytesIO

from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

TRANSFORM_TRAIN = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
TRANSFORM_EVAL = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

CLASES = sorted(train["categoria"].unique())
LABEL2ID = {c: i for i, c in enumerate(CLASES)}

class DatasetImagenes(Dataset):
    def __init__(self, df, transform):
        self.rutas = df["ruta_imagen"].tolist()
        self.etiquetas = df["categoria"].map(LABEL2ID).tolist()
        self.transform = transform

    def __len__(self):
        return len(self.etiquetas)

    def __getitem__(self, idx):
        img = Image.open(self.rutas[idx]).convert("RGB")
        return self.transform(img), self.etiquetas[idx]

BATCH_SIZE = 16
train_loader = DataLoader(DatasetImagenes(train, TRANSFORM_TRAIN), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(DatasetImagenes(val, TRANSFORM_EVAL), batch_size=BATCH_SIZE, num_workers=2)
test_loader = DataLoader(DatasetImagenes(test, TRANSFORM_EVAL), batch_size=BATCH_SIZE, num_workers=2)
print(f"{len(CLASES)} clases · {len(train)} train / {len(val)} val / {len(test)} test")


## 2. Función de entrenamiento genérica (dos fases + early stopping)

Fase 1: solo la cabeza (rápido, "aprende" a usar las features de
ImageNet). Fase 2: se descongela el último bloque y se reentrena con
learning rate más bajo (ajuste fino de verdad). Esto se repite igual para
ResNet-50 y EfficientNet-B0.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
from torch import nn
from torchvision import models


def evaluar_imagen(modelo, loader):
    modelo.eval()
    y_true, y_pred, perdida_total, n = [], [], 0.0, 0
    criterio = nn.CrossEntropyLoss()
    with torch.no_grad():
        for entradas, etiquetas in loader:
            entradas, etiquetas = entradas.to(DISPOSITIVO), etiquetas.to(DISPOSITIVO)
            logits = modelo(entradas)
            perdida_total += criterio(logits, etiquetas).item() * len(etiquetas)
            n += len(etiquetas)
            y_pred.extend(logits.argmax(dim=1).cpu().tolist())
            y_true.extend(etiquetas.cpu().tolist())
    return {
        "loss": perdida_total / max(n, 1),
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }


def entrenar_imagen(modelo, nombre, epocas_cabeza, epocas_finetune, carpeta_checkpoint):
    criterio = nn.CrossEntropyLoss()
    historia = {"epoca": [], "fase": [], "train_loss": [], "val_loss": [], "val_accuracy": [], "val_f1_macro": []}
    mejor_f1, mejor_estado = -1.0, None
    epoca_global = 0

    for fase, epocas, lr in [("cabeza", epocas_cabeza, 1e-3), ("fine-tune", epocas_finetune, 1e-4)]:
        if fase == "fine-tune":
            for nombre_param, parametro in modelo.named_parameters():
                if "layer4" in nombre_param or "features.8" in nombre_param or "fc" in nombre_param or "classifier" in nombre_param:
                    parametro.requires_grad = True
        parametros_entrenables = [p for p in modelo.parameters() if p.requires_grad]
        optimizador = torch.optim.AdamW(parametros_entrenables, lr=lr)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizador, step_size=max(epocas // 2, 1), gamma=0.5)
        epocas_sin_mejora = 0

        for _ in range(epocas):
            epoca_global += 1
            modelo.train()
            perdida_total, n = 0.0, 0
            for entradas, etiquetas in train_loader:
                entradas, etiquetas = entradas.to(DISPOSITIVO), etiquetas.to(DISPOSITIVO)
                optimizador.zero_grad()
                perdida = criterio(modelo(entradas), etiquetas)
                perdida.backward()
                optimizador.step()
                perdida_total += perdida.item() * len(etiquetas)
                n += len(etiquetas)
            scheduler.step()

            metricas_val = evaluar_imagen(modelo, val_loader)
            historia["epoca"].append(epoca_global)
            historia["fase"].append(fase)
            historia["train_loss"].append(perdida_total / max(n, 1))
            historia["val_loss"].append(metricas_val["loss"])
            historia["val_accuracy"].append(metricas_val["accuracy"])
            historia["val_f1_macro"].append(metricas_val["f1_macro"])
            print(f"[{nombre}] {fase} época {epoca_global}  train_loss={historia['train_loss'][-1]:.4f}  val_f1_macro={metricas_val['f1_macro']:.4f}")

            if metricas_val["f1_macro"] > mejor_f1:
                mejor_f1 = metricas_val["f1_macro"]
                mejor_estado = {k: v.cpu().clone() for k, v in modelo.state_dict().items()}
                epocas_sin_mejora = 0
                torch.save({"modelo": mejor_estado, "clases": CLASES}, carpeta_checkpoint / f"checkpoint_{nombre}.pt")
            else:
                epocas_sin_mejora += 1
                if epocas_sin_mejora >= 2:
                    print(f"Early stopping ({nombre}, fase {fase}).")
                    break

    modelo.load_state_dict(mejor_estado)
    return modelo, historia, mejor_f1


## 3. Entrenar ResNet-50

In [ ]:
EPOCAS_CABEZA = 1 if MODO_RAPIDO else 3
EPOCAS_FINETUNE = 1 if MODO_RAPIDO else 4

resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
for parametro in resnet.parameters():
    parametro.requires_grad = False
resnet.fc = nn.Linear(resnet.fc.in_features, len(CLASES))
resnet = resnet.to(DISPOSITIVO)

resnet, historia_resnet, f1_resnet = entrenar_imagen(resnet, "resnet50", EPOCAS_CABEZA, EPOCAS_FINETUNE, CARPETA_MODELOS)


## 4. Entrenar EfficientNet-B0

In [ ]:
efficientnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
for parametro in efficientnet.parameters():
    parametro.requires_grad = False
efficientnet.classifier[1] = nn.Linear(efficientnet.classifier[1].in_features, len(CLASES))
efficientnet = efficientnet.to(DISPOSITIVO)

efficientnet, historia_efficientnet, f1_efficientnet = entrenar_imagen(
    efficientnet, "efficientnet_b0", EPOCAS_CABEZA, EPOCAS_FINETUNE, CARPETA_MODELOS
)


## 5. Curvas y comparación de ambas arquitecturas

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for historia, nombre in [(historia_resnet, "ResNet-50"), (historia_efficientnet, "EfficientNet-B0")]:
    ax1.plot(historia["epoca"], historia["train_loss"], label=f"{nombre} train")
    ax2.plot(historia["epoca"], historia["val_f1_macro"], label=nombre)
ax1.set_title("Pérdida de entrenamiento"); ax1.set_xlabel("época"); ax1.legend()
ax2.set_title("F1-macro de validación"); ax2.set_xlabel("época"); ax2.legend()
plt.tight_layout()
plt.savefig(CARPETA_MODELOS / "curvas_entrenamiento.png", dpi=120)
plt.show()

metricas_resnet_test = evaluar_imagen(resnet, test_loader)
metricas_efficientnet_test = evaluar_imagen(efficientnet, test_loader)
tabla = pd.DataFrame({
    "modelo": ["ResNet-50", "EfficientNet-B0"],
    "val_f1_macro": [f1_resnet, f1_efficientnet],
    "test_accuracy": [metricas_resnet_test["accuracy"], metricas_efficientnet_test["accuracy"]],
    "test_f1_macro": [metricas_resnet_test["f1_macro"], metricas_efficientnet_test["f1_macro"]],
})
print(tabla.to_string(index=False))


## 6. Guardar ambos modelos + extractor de embeddings

In [ ]:
import json

for nombre, modelo, metricas_test, dim in [
    ("resnet50", resnet, metricas_resnet_test, 2048),
    ("efficientnet_b0", efficientnet, metricas_efficientnet_test, 1280),
]:
    torch.save({"modelo": modelo.state_dict(), "clases": CLASES, "arquitectura": nombre}, CARPETA_MODELOS / f"{nombre}.pt")
    with open(CARPETA_MODELOS / f"metricas_{nombre}.json", "w", encoding="utf-8") as f:
        json.dump(metricas_test, f, indent=2)
    print(f"{nombre}: dimensión de embedding = {dim}")

def extraer_embedding(modelo, imagen_pil):
    """Penúltima capa (avgpool/features), antes del clasificador final."""
    entrada = TRANSFORM_EVAL(imagen_pil).unsqueeze(0).to(DISPOSITIVO)
    extractor = nn.Sequential(*list(modelo.children())[:-1])
    modelo.eval()
    with torch.no_grad():
        salida = extractor(entrada)
    return salida.flatten(1).squeeze(0).cpu().tolist()

vector_prueba = extraer_embedding(resnet, Image.open(test["ruta_imagen"].iloc[0]).convert("RGB"))
print(f"Embedding ResNet-50 de prueba: {len(vector_prueba)} dimensiones (debe ser 2048).")
assert len(vector_prueba) == 2048


## 7. Fusión imagen + texto

Se usa el modelo de imagen con mejor F1-macro de validación (de los dos
de arriba) combinado con BETO (02_cu2_beto.ipynb, debe haber corrido
antes). El peso de la imagen en el promedio se barre entre 0 y 1 EN
VALIDACIÓN, y el que da mejor F1-macro es el que se reporta sobre
PRUEBA — así la elección del peso no "ve" el conjunto de prueba.

In [ ]:
mejor_modelo_imagen = resnet if f1_resnet >= f1_efficientnet else efficientnet
nombre_mejor = "ResNet-50" if f1_resnet >= f1_efficientnet else "EfficientNet-B0"
print(f"Modelo de imagen para la fusión: {nombre_mejor}")

ruta_beto = CARPETA_MODELOS.parent / "cu2-beto-v1"
if not ruta_beto.exists():
    print(f"No se encontró {ruta_beto} — corre 02_cu2_beto.ipynb primero para evaluar la fusión. Se omite esta sección.")
else:
    from transformers import AutoModelForSequenceClassification as AMSC
    from transformers import AutoTokenizer as AT

    tokenizer_beto = AT.from_pretrained(ruta_beto)
    modelo_beto = AMSC.from_pretrained(ruta_beto).to(DISPOSITIVO).eval()
    clases_beto = [modelo_beto.config.id2label[i] for i in range(len(modelo_beto.config.id2label))]

    def probs_imagen(loader_df):
        mejor_modelo_imagen.eval()
        todas = []
        with torch.no_grad():
            for _, fila in loader_df.iterrows():
                img = Image.open(fila["ruta_imagen"]).convert("RGB")
                entrada = TRANSFORM_EVAL(img).unsqueeze(0).to(DISPOSITIVO)
                probs = torch.softmax(mejor_modelo_imagen(entrada), dim=1).squeeze(0).cpu()
                todas.append(probs)
        return torch.stack(todas)

    def probs_texto(loader_df):
        todas = []
        with torch.no_grad():
            for texto in loader_df["texto_es"]:
                entrada = tokenizer_beto(texto, return_tensors="pt", truncation=True, max_length=128).to(DISPOSITIVO)
                logits = modelo_beto(**entrada).logits
                probs = torch.softmax(logits, dim=1).squeeze(0).cpu()
                todas.append(probs)
        return torch.stack(todas)

    # Asume que CLASES (orden del modelo de imagen) coincide con clases_beto —
    # ambos se entrenaron con sorted(df["categoria"].unique()) sobre la MISMA
    # partición, así que el orden debe ser idéntico; se valida explícitamente.
    assert CLASES == clases_beto, "Las clases de imagen y texto no coinciden — ¿misma partición?"

    probs_img_val, probs_txt_val = probs_imagen(val), probs_texto(val)
    y_val_idx = val["categoria"].map(LABEL2ID).tolist()

    mejor_peso, mejor_f1_fusion = 0.5, -1.0
    for peso in [i / 10 for i in range(0, 11)]:
        combinado = peso * probs_img_val + (1 - peso) * probs_txt_val
        pred = combinado.argmax(dim=1).tolist()
        f1 = f1_score(y_val_idx, pred, average="macro", zero_division=0)
        if f1 > mejor_f1_fusion:
            mejor_f1_fusion, mejor_peso = f1, peso
    print(f"Mejor peso de imagen en validación: {mejor_peso} (F1-macro val={mejor_f1_fusion:.4f})")

    probs_img_test, probs_txt_test = probs_imagen(test), probs_texto(test)
    y_test_idx = test["categoria"].map(LABEL2ID).tolist()
    combinado_test = mejor_peso * probs_img_test + (1 - mejor_peso) * probs_txt_test
    pred_test = combinado_test.argmax(dim=1).tolist()

    metricas_fusion = {
        "peso_imagen": mejor_peso,
        "accuracy": float(accuracy_score(y_test_idx, pred_test)),
        "f1_macro": float(f1_score(y_test_idx, pred_test, average="macro", zero_division=0)),
    }
    print("Fusión sobre PRUEBA:", json.dumps(metricas_fusion, indent=2))

    with open(CARPETA_MODELOS / "metricas_fusion.json", "w", encoding="utf-8") as f:
        json.dump(metricas_fusion, f, indent=2)


## 8. SQL para registrar los modelos en `modelos_ia`

In [ ]:
for nombre_bd, metricas_test in [
    ("cu1-resnet50-v1", metricas_resnet_test),
    ("cu1-efficientnet-b0-v1", metricas_efficientnet_test),
]:
    sql = f"""
UPDATE novastore.modelos_ia
   SET metricas = '{json.dumps(metricas_test)}'::jsonb,
       estado = 'evaluado',
       entrenado_en = now(),
       ruta_artefacto = '{nombre_bd}.pt'
 WHERE codigo = '{nombre_bd}';
"""
    (CARPETA_MODELOS / f"registro_modelo_{nombre_bd}.sql").write_text(sql, encoding="utf-8")
    print(sql)

cumple_oe1 = max(metricas_resnet_test["accuracy"], metricas_efficientnet_test["accuracy"]) >= 0.80 and \
             max(metricas_resnet_test["f1_macro"], metricas_efficientnet_test["f1_macro"]) >= 0.75
print("Meta OE1 (accuracy>=0.80 y F1-macro>=0.75, mejor de los dos modelos):", "CUMPLIDA" if cumple_oe1 else "NO alcanzada todavía")
